# NIFTY Gap Strategy — v6 (Logistic Regression Signal Backtest)

**Key change from v5:** Replace exhaustive combo grid-search with a single logistic regression model.

| v5 approach | v6 approach |
|---|---|
| Test 1092 signal combos → Bonferroni kills all | One model with 13 signals → p-values valid at p < 0.05 |
| AND logic forces tiny N per combo | All 361 training rows contribute to every coefficient |
| Binary: combo fires or not | Continuous P(win) score → tunable threshold |
| Fallback always triggered | Significant signals likely found |

**Workflow:**
1. Simulate ALL 2024–2025 tradeable days → record actual trade outcomes
2. Fit logistic regression: P(TP hit) ~ 13 binary signals
3. Inspect odds ratios + p-values — no multiple testing correction needed
4. Sweep probability thresholds on training data to find a good cutoff
5. Apply model + threshold to 2026 (OOS) with compounding capital

| Parameter | Value |
|---|---|
| SL | −15% of entry premium |
| TP | +40% of entry premium |
| Hard exit | 11:15 AM IST |
| Strike | ATM − 50 (1-OTM PUT) |
| Training | 2024 – 2025 |
| OOS | 2026+ |

In [6]:
import pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date, timedelta
import statsmodels.api as sm
import yfinance as yf

warnings.filterwarnings('ignore')

# ── Trade parameters ──────────────────────────────────────────────────────────
SL_PCT           = 0.15
TP_PCT           = 0.40
LOT_SIZE         = 75
STRIKE_STEP      = 50
BASE_LOTS        = 5
MAX_LOTS         = 25
DTE0_MAX_LOTS    = 10
STARTING_CAPITAL = 200_000.0
GAP_THR          = 0.0015
GAP_LARGE        = 0.0050
VIX_RISING_THR   = 0.03
VIX_SPIKE_THR    = 0.05
MAX_STALE_DAYS   = 5

# ── Walk-forward split ────────────────────────────────────────────────────────
TRAIN_END  = date(2025, 6, 30)   # train on 2024 + first half 2025
OOS_START  = date(2025, 7, 1)    # OOS = second half 2025 + 2026

# ── Logistic regression threshold ─────────────────────────────────────────────
PROB_THRESHOLD    = None   # None = auto-select from training data
                           # Set to float (e.g. 0.32) to override
MIN_THRESH_TRADES = 15     # auto-select needs at least this many training trades
EDGE_TARGET_PP    = 8      # auto-select targets base_win_rate + this many pp

OLD_CUTOFF = date(2024, 11, 1)
_MON = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

print('Config loaded.')
print(f'Train : 2024 – {TRAIN_END}  |  OOS : {OOS_START}+')
print(f'SL={SL_PCT:.0%}  TP={TP_PCT:.0%}')
print(f'Probability threshold : {"auto" if PROB_THRESHOLD is None else PROB_THRESHOLD}')


Config loaded.
Train : 2024 – 2025-06-30  |  OOS : 2025-07-01+
SL=15%  TP=40%
Probability threshold : auto


In [7]:
# ── Paths ─────────────────────────────────────────────────────────────────────
GAP_TRADING     = Path.cwd().parent
MARKET_RESEARCH = Path.cwd().parent.parent
ALIGNED_CSV     = GAP_TRADING / 'v2' / 'v2_aligned_dataset.csv'
MINUTE_CACHE    = GAP_TRADING / 'kite_minute_cache'
MERGE_OLD       = MARKET_RESEARCH / 'merged' / 'old_format'
MERGE_NEW       = MARKET_RESEARCH / 'merged' / 'expiry_wise'

for lbl, p in [('Aligned CSV', ALIGNED_CSV), ('Minute cache', MINUTE_CACHE),
               ('Merge old', MERGE_OLD), ('Merge new', MERGE_NEW)]:
    print(f'{lbl:<14}: {"OK" if p.exists() else "MISSING"}  ({p})')

# ── Load aligned dataset ──────────────────────────────────────────────────────
aligned = pd.read_csv(ALIGNED_CSV, parse_dates=['india_date'])
aligned = aligned.sort_values('india_date').reset_index(drop=True)
aligned['VIX_INDIA_level'] = aligned['VIX_INDIA_level'].ffill().bfill()
print(f'\nAligned: {aligned.india_date.min().date()} to {aligned.india_date.max().date()}  ({len(aligned)} rows)')

# ── Build spot map: date → NIFTY spot at 09:25 ───────────────────────────────
spot_map = {}
for pkl_path in sorted(MINUTE_CACHE.glob('minute_256265_*.pkl')):
    with open(pkl_path, 'rb') as f:
        chunk = pickle.load(f)
    chunk.index = pd.to_datetime(chunk.index)
    if chunk.index.tzinfo is not None:
        chunk.index = chunk.index.tz_localize(None)
    for dt, row in chunk.iterrows():
        if dt.strftime('%H:%M') == '09:25':
            spot_map[dt.date()] = float(row['open'])
print(f'Spot map: {len(spot_map)} dates with 09:25 NIFTY level')

# ── Fetch ^N225 ───────────────────────────────────────────────────────────────
ds_start = aligned['india_date'].min().date() - timedelta(days=10)
ds_end   = aligned['india_date'].max().date() + timedelta(days=2)
print(f'\nFetching ^N225 {ds_start} to {ds_end} ...')
try:
    n225 = yf.download('^N225', start=str(ds_start), end=str(ds_end),
                       progress=False, auto_adjust=True)
    if isinstance(n225.columns, pd.MultiIndex):
        n225.columns = n225.columns.get_level_values(0)
    n225.index = pd.to_datetime(n225.index)
    if n225.index.tzinfo is None:
        n225.index = n225.index.tz_localize('UTC')
    n225_ok = len(n225) > 0
    print(f'N225: {len(n225)} rows  ({n225.index[0].date()} to {n225.index[-1].date()})')
except Exception as e:
    n225 = pd.DataFrame(); n225_ok = False
    print(f'WARNING: ^N225 fetch failed: {e}')

def get_n225_sgx_ret(india_date: date):
    if not n225_ok or len(n225) == 0:
        return None
    past_rows  = n225[n225.index.date < india_date]
    today_rows = n225[n225.index.date == india_date]
    if len(past_rows) < 2:
        return None
    close_prev = float(past_rows['Close'].iloc[-1])
    prev_date  = past_rows.index[-1].date()
    if (india_date - prev_date).days > MAX_STALE_DAYS:
        return None
    if len(today_rows) > 0:
        return (float(today_rows['Open'].iloc[0]) - close_prev) / close_prev
    else:
        close_d2 = float(past_rows['Close'].iloc[-2])
        return (close_prev - close_d2) / close_d2

aligned['sgx_ret_v5'] = aligned['india_date'].dt.date.map(get_n225_sgx_ret)
print(f'SGX signal: {aligned["sgx_ret_v5"].notna().sum()}/{len(aligned)} dates have valid N225 data')

Aligned CSV   : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v2\v2_aligned_dataset.csv)
Minute cache  : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\kite_minute_cache)
Merge old     : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\merged\old_format)
Merge new     : OK  (c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\merged\expiry_wise)

Aligned: 2023-03-31 to 2026-04-02  (740 rows)
Spot map: 742 dates with 09:25 NIFTY level

Fetching ^N225 2023-03-21 to 2026-04-04 ...
N225: 743 rows  (2023-03-22 to 2026-04-03)
SGX signal: 736/740 dates have valid N225 data


In [8]:
def d2dmy(d: date) -> str:
    return f'{d.day:02d}{_MON[d.month-1]}{str(d.year)[2:]}'

def _load_old(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    mon = _MON[trade_date.month - 1]
    fp  = MERGE_OLD / f'2024{mon}' / f'NIFTY-{d2dmy(expiry_date)}-{d2dmy(trade_date)}.csv'
    if not fp.exists():
        return None
    df = pd.read_csv(fp)
    df.columns = [c.strip() for c in df.columns]
    df['time_str'] = df['datetime'].astype(str).str[:5]
    return df

def _load_new(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    exp_dir = MERGE_NEW / expiry_date.strftime('%Y-%m-%d')
    if not exp_dir.exists():
        return None
    exp_str = f'{expiry_date.day:02d}_{_MON[expiry_date.month-1]}_{str(expiry_date.year)[2:]}'
    files   = list(exp_dir.glob(f'NIFTY_*_*_{exp_str}.csv'))
    if not files:
        return None
    dfs = []
    for fp in files:
        parts = fp.stem.split('_')
        try:
            strike = int(parts[1]); right = parts[2]
        except (ValueError, IndexError):
            continue
        try:
            df = pd.read_csv(fp, usecols=['timestamp','open','high','low','close','volume','oi'])
        except Exception:
            continue
        df['ts'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
        day = df[df['ts'].dt.date == trade_date].copy()
        if day.empty:
            continue
        day['time_str']     = day['ts'].dt.strftime('%H:%M')
        day['strike_price'] = strike
        day['right']        = right
        day.rename(columns={'oi': 'open_interest'}, inplace=True)
        dfs.append(day[['time_str','strike_price','right','open','high','low','close']])
    if not dfs:
        return None
    return (pd.concat(dfs)
              .sort_values(['time_str','strike_price','right'])
              .reset_index(drop=True))

_opt_cache: dict = {}

def load_opt(trade_date: date, expiry_date: date) -> pd.DataFrame | None:
    k = (trade_date, expiry_date)
    if k not in _opt_cache:
        _opt_cache[k] = (_load_old(trade_date, expiry_date)
                         if expiry_date < OLD_CUTOFF
                         else _load_new(trade_date, expiry_date))
    return _opt_cache[k]

print('Option loaders ready.')

Option loaders ready.


In [9]:
# ── NSE helpers ───────────────────────────────────────────────────────────────
NSE_HOLIDAYS = {
    date(2024,  1, 22), date(2024,  3, 25), date(2024,  3, 29), date(2024,  4, 14),
    date(2024,  4, 17), date(2024,  5, 23), date(2024,  6, 17), date(2024,  7, 17),
    date(2024,  8, 15), date(2024, 10,  2), date(2024, 10, 24), date(2024, 11,  1),
    date(2024, 11, 15), date(2024, 12, 25),
    date(2025,  2, 26), date(2025,  3, 14), date(2025,  3, 31), date(2025,  4, 10),
    date(2025,  4, 14), date(2025,  4, 18), date(2025,  5,  1), date(2025,  8, 15),
    date(2025,  8, 27), date(2025, 10,  2), date(2025, 10, 21), date(2025, 10, 22),
    date(2025, 11,  5), date(2025, 12, 25),
    date(2026,  1, 26), date(2026,  3, 26),
}
EVENT_DAYS = {
    date(2024,  2,  1), date(2024,  2,  8), date(2024,  4,  5),
    date(2024,  6,  7), date(2024,  8,  8), date(2024, 10,  9), date(2024, 12,  6),
    date(2025,  2,  1), date(2025,  2,  7), date(2025,  4,  9),
    date(2025,  6,  6), date(2025,  8,  6), date(2025, 10,  8), date(2025, 12,  5),
    date(2026,  2,  1), date(2026,  2,  6), date(2026,  4,  8),
    date(2026,  6,  5), date(2026,  8,  7), date(2026, 10,  7), date(2026, 12,  4),
}
EXPIRY_CHANGE = date(2025, 9, 2)

def is_skip_day(d: date) -> bool:
    return d.weekday() == 0 or d in NSE_HOLIDAYS or d in EVENT_DAYS

def next_expiry(d: date) -> date:
    wd = 1 if d >= EXPIRY_CHANGE else 3
    return d + timedelta(days=(wd - d.weekday()) % 7)

# ── 13 binary signal definitions ─────────────────────────────────────────────
SIGNAL_NAMES = [
    'Gap Up', 'Gap Up Strong', 'Gap Down',
    'Prev India UP', 'Prev India DOWN',
    'US UP', 'US DOWN',
    'SGX UP', 'SGX DOWN',
    'DAX UP',
    'VIX Rising', 'VIX Falling', 'VIX Spike',
]

def compute_signals_v5(row) -> dict:
    gap = float(row['gap_pct']) if pd.notna(row.get('gap_pct', float('nan'))) else 0.0
    def _f(col):
        v = row.get(col)
        return float(v) if pd.notna(v) else None
    sgx  = _f('sgx_ret_v5')
    sp5  = _f('SP500_ret')
    dax  = _f('DAX_ret')
    vix  = _f('VIX_US_ret')
    prev = _f('prev_india_ret')
    return {
        'Gap Up'         : gap >  GAP_THR,
        'Gap Up Strong'  : gap >  GAP_LARGE,
        'Gap Down'       : gap < -GAP_THR,
        'Prev India UP'  : prev is not None and prev > 0,
        'Prev India DOWN': prev is not None and prev < 0,
        'US UP'          : sp5  is not None and sp5  > 0,
        'US DOWN'        : sp5  is not None and sp5  < 0,
        'SGX UP'         : sgx  is not None and sgx  > 0,
        'SGX DOWN'       : sgx  is not None and sgx  < 0,
        'DAX UP'         : dax  is not None and dax  > 0,
        'VIX Rising'     : vix  is not None and vix  > VIX_RISING_THR,
        'VIX Falling'    : vix  is not None and vix  < 0,
        'VIX Spike'      : vix  is not None and vix  > VIX_SPIKE_THR,
    }

# ── Trade simulation ──────────────────────────────────────────────────────────
def round_trip_charges(entry_prem: float, exit_prem: float, lots: int) -> float:
    buy_val  = entry_prem * lots * LOT_SIZE
    sell_val = exit_prem  * lots * LOT_SIZE
    brok  = 20.0 * 2
    stamp = 0.00003  * buy_val
    stt   = 0.000625 * sell_val
    exch  = 0.00053  * (buy_val + sell_val)
    sebi  = 0.000001 * (buy_val + sell_val)
    gst   = 0.18     * (brok + exch + sebi)
    return round(brok + stamp + stt + exch + sebi + gst, 2)

def simulate_trade_real(d: date) -> dict | None:
    spot_925 = spot_map.get(d)
    if spot_925 is None:
        return None
    atm    = round(spot_925 / STRIKE_STEP) * STRIKE_STEP
    strike = atm - STRIKE_STEP
    expiry = next_expiry(d)
    dte    = (expiry - d).days
    opt_df = load_opt(d, expiry)
    if opt_df is None:
        return None
    pe = opt_df[(opt_df['strike_price'] == strike) & (opt_df['right'] == 'PE')].copy()
    if pe.empty:
        pe = opt_df[(opt_df['strike_price'] == atm) & (opt_df['right'] == 'PE')].copy()
        if pe.empty:
            return None
        strike = atm
    pe = pe.sort_values('time_str').reset_index(drop=True)
    entry_row = pe[pe['time_str'] == '09:25']
    if entry_row.empty:
        return None
    entry_prem = float(entry_row.iloc[0]['open'])
    if entry_prem < 0.5:
        return None
    sl_px = entry_prem * (1 - SL_PCT)
    tp_px = entry_prem * (1 + TP_PCT)
    monitor = pe[(pe['time_str'] >= '09:26') & (pe['time_str'] <= '11:15')]
    exit_prem, exit_reason, exit_time = None, '11:15 exit', '11:15'
    for _, row in monitor.iterrows():
        lo, hi, t = float(row['low']), float(row['high']), row['time_str']
        if lo <= sl_px:
            exit_prem, exit_reason, exit_time = sl_px, 'Stop Loss', t
            break
        if hi >= tp_px:
            exit_prem, exit_reason, exit_time = tp_px, 'Target Hit', t
            break
    if exit_prem is None:
        exit_row = pe[pe['time_str'] == '11:15']
        if not exit_row.empty:
            exit_prem = float(exit_row.iloc[0]['close'])
        elif not monitor.empty:
            exit_prem = float(monitor.iloc[-1]['close'])
        else:
            exit_prem = entry_prem
    return {
        'expiry': expiry, 'dte': dte, 'atm': int(atm), 'strike': int(strike),
        'entry_prem': round(entry_prem, 2), 'exit_prem': round(exit_prem, 2),
        'pnl_pts': round(exit_prem - entry_prem, 2),
        'exit_reason': exit_reason, 'exit_time': exit_time,
    }

print('NSE helpers, signals, and simulation ready.')

NSE helpers, signals, and simulation ready.


In [10]:
# ── Simulation cache ──────────────────────────────────────────────────────────
# First run: simulates every available trading day and saves to CSV (~2-3 min).
# Subsequent runs: loads instantly. Delete sim_cache.csv to force a re-run.
# ─────────────────────────────────────────────────────────────────────────────
CACHE_PATH = Path.cwd() / 'sim_cache.csv'

if CACHE_PATH.exists():
    print(f'Cache found — loading {CACHE_PATH.name} ...')
    sim_df = pd.read_csv(CACHE_PATH, parse_dates=['date'])
    sim_df['date'] = sim_df['date'].dt.date
    for s in SIGNAL_NAMES:
        sim_df[s] = sim_df[s].astype(bool)
    print(f'Loaded {len(sim_df)} days  ({sim_df["date"].min()} → {sim_df["date"].max()})')
else:
    print('No cache found. Simulating ALL available trading days (takes ~2–3 min)...')
    all_rows, no_data = [], 0

    for _, row in aligned.iterrows():
        d = row['india_date'].date()
        if is_skip_day(d):
            continue
        sigs = compute_signals_v5(row)
        res  = simulate_trade_real(d)
        if res is None:
            no_data += 1
            continue
        r = {
            'date'       : d,
            'win'        : res['exit_reason'] == 'Target Hit',
            'exit_reason': res['exit_reason'],
            'entry_prem' : res['entry_prem'],
            'exit_prem'  : res['exit_prem'],
            'dte'        : res['dte'],
        }
        r.update(sigs)
        all_rows.append(r)

    sim_df = pd.DataFrame(all_rows)
    sim_df.to_csv(CACHE_PATH, index=False)
    print(f'Done. {len(sim_df)} days saved → {CACHE_PATH}')
    print(f'(Missing option data skipped: {no_data})')

# ── Train / OOS split ─────────────────────────────────────────────────────────
train_df = sim_df[sim_df['date'] <= TRAIN_END].reset_index(drop=True)
oos_df   = sim_df[sim_df['date'] >= OOS_START].reset_index(drop=True)

base_win_rate = train_df['win'].mean()

print(f'\nTrain : {train_df["date"].min()} → {train_df["date"].max()}  ({len(train_df)} days)')
print(f'OOS   : {oos_df["date"].min()} → {oos_df["date"].max()}  ({len(oos_df)} days)')
print(f'\nTraining outcomes:')
print(f'  Base win rate : {base_win_rate:.1%}')
print(train_df['exit_reason'].value_counts().to_string())


No cache found. Simulating ALL available trading days (takes ~2–3 min)...
Done. 402 days saved → c:\Users\sayan\OneDrive\Desktop\Projects\03_Market_Research\market-research\gap_trading\v6\sim_cache.csv
(Missing option data skipped: 174)

Train : 2024-01-02 → 2025-06-27  (269 days)
OOS   : 2025-07-01 → 2026-03-24  (133 days)

Training outcomes:
  Base win rate : 23.8%
exit_reason
Stop Loss     192
Target Hit     64
11:15 exit     13


In [11]:
# ── Signal correlation matrix ─────────────────────────────────────────────────
sig_mat = train_df[SIGNAL_NAMES].astype(int)
corr = sig_mat.corr().round(2)

print('Signal correlation matrix (training data):')
print('(Pairs near ±1.0 are collinear — coefficients may be less stable)')
print()
print(corr.to_string())
print()

# Flag highly correlated pairs
high_corr = []
for i in range(len(SIGNAL_NAMES)):
    for j in range(i+1, len(SIGNAL_NAMES)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.6:
            high_corr.append((SIGNAL_NAMES[i], SIGNAL_NAMES[j], r))
if high_corr:
    print('High-correlation pairs (|r| >= 0.6):')
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f'  {a:<22} x  {b:<22}  r={r:+.2f}')
else:
    print('No pairs with |r| >= 0.6')

# ── Fit logistic regression ───────────────────────────────────────────────────
print('\n' + '='*60)
print('  LOGISTIC REGRESSION FIT')
print('='*60)

X_train = sm.add_constant(sig_mat.copy())
y_train = train_df['win'].astype(int)

try:
    logit_model = sm.Logit(y_train, X_train).fit(maxiter=200, disp=True)
    print(logit_model.summary())
except Exception as e:
    print(f'ERROR fitting model: {e}')
    raise

Signal correlation matrix (training data):
(Pairs near ±1.0 are collinear — coefficients may be less stable)

                 Gap Up  Gap Up Strong  Gap Down  Prev India UP  Prev India DOWN  US UP  US DOWN  SGX UP  SGX DOWN  DAX UP  VIX Rising  VIX Falling  VIX Spike
Gap Up             1.00           0.40     -0.47           0.11            -0.10   0.26    -0.24    0.26     -0.27    0.25       -0.21         0.22      -0.14
Gap Up Strong      0.40           1.00     -0.19          -0.03             0.03   0.19    -0.16    0.11     -0.10    0.07       -0.08         0.19      -0.03
Gap Down          -0.47          -0.19      1.00          -0.17             0.16  -0.23     0.25   -0.25      0.25   -0.17        0.30        -0.22       0.29
Prev India UP      0.11          -0.03     -0.17           1.00            -0.99   0.18    -0.18    0.08     -0.07    0.15       -0.14         0.06      -0.14
Prev India DOWN   -0.10           0.03      0.16          -0.99             1.00  -0.17     0.1

In [12]:
# ── Odds ratio table ──────────────────────────────────────────────────────────
params   = logit_model.params.drop('const', errors='ignore')
pvalues  = logit_model.pvalues.drop('const', errors='ignore')
conf     = logit_model.conf_int().drop('const', errors='ignore')

odds_df = pd.DataFrame({
    'Coef'       : params.values.round(3),
    'Odds_Ratio' : np.exp(params.values).round(3),
    'CI_95_lo'   : np.exp(conf.iloc[:, 0].values).round(3),
    'CI_95_hi'   : np.exp(conf.iloc[:, 1].values).round(3),
    'p_val'      : pvalues.values.round(4),
}, index=params.index)

odds_df['p<0.05'] = odds_df['p_val'] < 0.05
odds_df['p<0.10'] = odds_df['p_val'] < 0.10
odds_df = odds_df.sort_values('Odds_Ratio', ascending=False)

print('Odds Ratios — Odds Ratio > 1.0 means signal INCREASES P(win)')
print('No Bonferroni needed: one model, not many separate tests')
print()
print(odds_df.to_string())
print()

bullish_sig = odds_df[(odds_df['p_val'] < 0.05) & (odds_df['Odds_Ratio'] > 1.0)].index.tolist()
bearish_sig = odds_df[(odds_df['p_val'] < 0.05) & (odds_df['Odds_Ratio'] < 1.0)].index.tolist()

print(f'Significant signals that INCREASE P(win) [p<0.05, OR>1]: {bullish_sig or "none"}')
print(f'Significant signals that DECREASE P(win) [p<0.05, OR<1]: {bearish_sig or "none"}')

if odds_df['p_val'].min() > 0.10:
    print('\nWARNING: No individual signal is significant at p<0.10.')
    print('The model may not have enough data to isolate independent signal effects.')

Odds Ratios — Odds Ratio > 1.0 means signal INCREASES P(win)
No Bonferroni needed: one model, not many separate tests

                   Coef    Odds_Ratio  CI_95_lo  CI_95_hi   p_val  p<0.05  p<0.10
SGX DOWN         31.106  3.229862e+13     0.000       inf  1.0000   False   False
SGX UP           30.873  2.557257e+13     0.000       inf  1.0000   False   False
VIX Spike         1.812  6.124000e+00     0.708    52.944  0.0996   False    True
Gap Up Strong     0.738  2.092000e+00     0.836     5.236  0.1147   False   False
US UP             0.659  1.933000e+00     0.336    11.130  0.4607   False   False
Gap Up            0.533  1.704000e+00     0.789     3.679  0.1748   False   False
Gap Down          0.234  1.263000e+00     0.526     3.031  0.6007   False   False
DAX UP            0.094  1.098000e+00     0.572     2.110  0.7788   False   False
US DOWN          -0.088  9.150000e-01     0.156     5.379  0.9221   False   False
VIX Falling      -0.178  8.370000e-01     0.355     1.976  0.

In [13]:
# ── Predicted probabilities on training data ──────────────────────────────────
train_df['prob_win'] = logit_model.predict(X_train)

print(f'Predicted P(win) distribution on training data:')
print(f'  Min  : {train_df["prob_win"].min():.3f}')
print(f'  Mean : {train_df["prob_win"].mean():.3f}  (= base win rate {base_win_rate:.1%} by construction)')
print(f'  Max  : {train_df["prob_win"].max():.3f}')
print()

# ── Threshold sweep ───────────────────────────────────────────────────────────
thresholds = np.arange(0.24, 0.52, 0.02).round(2)
rows = []
for thr in thresholds:
    sub = train_df[train_df['prob_win'] >= thr]
    n   = len(sub)
    if n == 0:
        continue
    wins     = int(sub['win'].sum())
    win_rate = wins / n
    rows.append({'Threshold': thr, 'Trades': n, 'Wins': wins,
                 'Win%': round(win_rate * 100, 1),
                 'Edge_pp': round((win_rate - base_win_rate) * 100, 1)})

thr_df = pd.DataFrame(rows)
print('Threshold sweep on TRAINING data (in-sample — use to pick a cutoff, not to judge performance):')
print(thr_df.to_string(index=False))
print()

# ── Auto-select threshold ─────────────────────────────────────────────────────
target_wr = base_win_rate + EDGE_TARGET_PP / 100
candidates = thr_df[
    (thr_df['Win%'] >= target_wr * 100) &
    (thr_df['Trades'] >= MIN_THRESH_TRADES)
]

if len(candidates) > 0:
    auto_thr = float(candidates['Threshold'].iloc[-1])
else:
    # Fallback: lowest threshold that still beats base rate with >=10 trades
    fallback = thr_df[(thr_df['Edge_pp'] > 0) & (thr_df['Trades'] >= 10)]
    auto_thr = float(fallback['Threshold'].iloc[-1]) if len(fallback) > 0 else 0.28
    print(f'WARNING: No threshold achieves +{EDGE_TARGET_PP}pp with {MIN_THRESH_TRADES}+ trades.')
    print(f'Falling back to threshold={auto_thr} (positive edge, 10+ trades).\n')

if PROB_THRESHOLD is None:
    selected_thr = auto_thr
    print(f'Auto-selected threshold : {selected_thr}  '
          f'(target edge +{EDGE_TARGET_PP}pp, min {MIN_THRESH_TRADES} trades)')
else:
    selected_thr = PROB_THRESHOLD
    print(f'Manually set threshold  : {selected_thr}')

ref = thr_df[thr_df['Threshold'] == selected_thr]
if not ref.empty:
    r = ref.iloc[0]
    print(f'At this threshold — Training: {int(r["Trades"])} trades, '
          f'{r["Win%"]:.1f}% win rate, edge={r["Edge_pp"]:+.1f}pp')

Predicted P(win) distribution on training data:
  Min  : 0.000
  Mean : 0.238  (= base win rate 23.8% by construction)
  Max  : 1.000



Threshold sweep on TRAINING data (in-sample — use to pick a cutoff, not to judge performance):
 Threshold  Trades  Wins  Win%  Edge_pp
      0.24     125    39  31.2      7.4
      0.26     100    34  34.0     10.2
      0.28      66    27  40.9     17.1
      0.30      53    22  41.5     17.7
      0.32      45    21  46.7     22.9
      0.34      36    20  55.6     31.8
      0.36      34    18  52.9     29.1
      0.38      30    16  53.3     29.5
      0.40      29    15  51.7     27.9
      0.42      29    15  51.7     27.9
      0.44      18    11  61.1     37.3
      0.46      17    10  58.8     35.0
      0.48      15     9  60.0     36.2
      0.50       9     5  55.6     31.8
      0.52       3     2  66.7     42.9

Auto-selected threshold : 0.48  (target edge +8pp, min 15 trades)
At this threshold — Training: 15 trades, 60.0% win rate, edge=+36.2pp


In [14]:
def run_oos_logit(period_label: str, sim_subset: pd.DataFrame):
    '''Run backtest using cached simulation data + logit model probabilities.
    No re-simulation — entry/exit prices come from sim_cache.csv.'''

    # Compute predicted probabilities for the whole subset at once (fast)
    X_sub = sm.add_constant(sim_subset[SIGNAL_NAMES].astype(int))
    probs = logit_model.predict(X_sub).values

    results, skipped_prob = [], 0
    for i, (_, row) in enumerate(sim_subset.iterrows()):
        prob = float(probs[i])
        if prob < selected_thr:
            skipped_prob += 1
            continue
        ep = float(row['entry_prem'])
        xp = float(row['exit_prem'])
        results.append({
            'Date'       : row['date'],
            'P(win)'     : round(prob, 3),
            'DTE'        : int(row['dte']),
            'Entry (pts)': ep,
            'Exit (pts)' : xp,
            'PnL (pts)'  : round(xp - ep, 2),
            'Exit Reason': row['exit_reason'],
        })

    if not results:
        print(f'{period_label}: no trades (threshold too high).')
        return pd.DataFrame()

    res_df = pd.DataFrame(results)

    capital, peak, ledger_rows = STARTING_CAPITAL, STARTING_CAPITAL, []
    for _, row in res_df.iterrows():
        ep, xp   = float(row['Entry (pts)']), float(row['Exit (pts)'])
        pnl_pts  = float(row['PnL (pts)'])
        dte      = int(row['DTE'])
        cost_lot = ep * LOT_SIZE
        if cost_lot <= 0:
            continue
        lots = max(BASE_LOTS, int(capital // cost_lot))
        lots = min(lots, MAX_LOTS)
        if dte == 0:
            lots = min(lots, DTE0_MAX_LOTS)
        charges   = round_trip_charges(ep, xp, lots)
        trade_pnl = pnl_pts * LOT_SIZE * lots - charges
        capital  += trade_pnl
        peak      = max(peak, capital)
        ledger_rows.append({
            'Trade#'     : len(ledger_rows) + 1,
            'Date'       : row['Date'],
            'P(win)'     : row['P(win)'],
            'DTE'        : dte,
            'Lots'       : lots,
            'Entry (pts)': ep,
            'Exit (pts)' : xp,
            'PnL (pts)'  : pnl_pts,
            'Charges'    : charges,
            'Trade PnL'  : round(trade_pnl, 2),
            'Capital'    : round(capital, 2),
            'Drawdown%'  : round((peak - capital) / peak * 100, 2),
            'Exit Reason': row['Exit Reason'],
        })

    ledger   = pd.DataFrame(ledger_rows)
    wins     = (ledger['Trade PnL'] > 0).sum()
    total    = len(ledger)
    roi      = (capital - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    max_dd   = ledger['Drawdown%'].max()
    avg_win  = ledger.loc[ledger['Trade PnL'] > 0,  'Trade PnL'].mean() if wins > 0     else 0.0
    avg_loss = ledger.loc[ledger['Trade PnL'] <= 0, 'Trade PnL'].mean() if wins < total else 0.0

    print(f'\n{"="*58}')
    print(f'  {period_label}')
    print(f'{"="*58}')
    print(f'  Below threshold (not traded)  : {skipped_prob}')
    print(f'  Above threshold (traded)      : {total}')
    print(f'  Win rate                      : {wins/total*100:.1f}%  (base: {base_win_rate:.1%})')
    print(f'  ROI                           : {roi:+.1f}%')
    print(f'  Max drawdown                  : {max_dd:.1f}%')
    print(f'  Avg win / loss                : Rs {avg_win:,.0f} / Rs {avg_loss:,.0f}')
    print(f'{"="*58}')
    print(ledger['Exit Reason'].value_counts().to_string())
    print()
    print(ledger[['Trade#','Date','P(win)','DTE','Lots','Entry (pts)',
                  'Exit (pts)','PnL (pts)','Trade PnL','Capital',
                  'Drawdown%','Exit Reason']].to_string(index=False))
    return ledger


# ── OOS: Jul 2025 → end of data ───────────────────────────────────────────────
ledger_oos = run_oos_logit(
    f'OOS  {OOS_START} → {oos_df["date"].max()} (out-of-sample)', oos_df)



  OOS  2025-07-01 → 2026-03-24 (out-of-sample)
  Below threshold (not traded)  : 128
  Above threshold (traded)      : 5
  Win rate                      : 40.0%  (base: 23.8%)
  ROI                           : -2.2%
  Max drawdown                  : 14.9%
  Avg win / loss                : Rs 26,596 / Rs -19,199
Exit Reason
Stop Loss     2
Target Hit    2
11:15 exit    1

 Trade#       Date  P(win)  DTE  Lots  Entry (pts)  Exit (pts)  PnL (pts)  Trade PnL   Capital  Drawdown% Exit Reason
      1 2026-01-22   0.485    5    24       109.15       92.78     -16.37  -29851.22 170148.78      14.93   Stop Loss
      2 2026-02-03   0.509    0    10       113.10      158.34      45.24   33678.47 203827.25       0.00  Target Hit
      3 2026-03-05   0.509    5    12       211.40      191.10     -20.30  -18657.38 185169.87       9.15  11:15 exit
      4 2026-03-10   0.485    0    10        65.60       91.84      26.24   19514.29 204684.16       0.00  Target Hit
      5 2026-03-24   0.509    0    

In [15]:
# ── In-sample reference ───────────────────────────────────────────────────────
ledger_train = run_oos_logit(
    f'TRAIN 2024 – {TRAIN_END} (in-sample reference)', train_df)

# ── Summary table ─────────────────────────────────────────────────────────────
def _summarize(ledger, label):
    if ledger.empty:
        return {'Period': label, 'Trades': 0, 'Win%': 'N/A', 'ROI': 'N/A', 'MaxDD': 'N/A'}
    wins  = (ledger['Trade PnL'] > 0).sum()
    total = len(ledger)
    roi   = (ledger['Capital'].iloc[-1] - STARTING_CAPITAL) / STARTING_CAPITAL * 100
    maxdd = ledger['Drawdown%'].max()
    return {'Period': label, 'Trades': total,
            'Win%' : f'{wins/total*100:.1f}%',
            'ROI'  : f'{roi:+.1f}%',
            'MaxDD': f'{maxdd:.1f}%'}

summary = pd.DataFrame([
    _summarize(ledger_train, f'TRAIN 2024–Jun 2025   (in-sample)'),
    _summarize(ledger_oos,   f'OOS   Jul 2025–{oos_df["date"].max()}  (out-of-sample)'),
])

print()
print('=' * 64)
print('  v6 LOGISTIC REGRESSION SUMMARY')
print('=' * 64)
print(summary.to_string(index=False))
print('=' * 64)
print('  v5 OOS 2026 (combo fallback)  :  10 trades | 20.0% win | -37.1% ROI')
print('  v43 reference (in-sample)     : 120 trades | 33.3% win | +147.7% ROI')
print('=' * 64)
print()
print(f'  Prob threshold (used)         : {selected_thr}')
print(f'  Base train win rate           : {base_win_rate:.1%}')
print(f'  Training days                 : {len(train_df)}')
print(f'  OOS days                      : {len(oos_df)}')
print(f'  Sig signals (p<0.05, OR>1)    : {bullish_sig or "none"}')
print(f'  Sig signals (p<0.05, OR<1)    : {bearish_sig or "none"}')



  TRAIN 2024 – 2025-06-30 (in-sample reference)
  Below threshold (not traded)  : 254
  Above threshold (traded)      : 15
  Win rate                      : 60.0%  (base: 23.8%)
  ROI                           : +117.1%
  Max drawdown                  : 27.5%
  Avg win / loss                : Rs 46,254 / Rs -30,362
Exit Reason
Target Hit    9
Stop Loss     6

 Trade#       Date  P(win)  DTE  Lots  Entry (pts)  Exit (pts)  PnL (pts)  Trade PnL   Capital  Drawdown% Exit Reason
      1 2024-01-09   0.509    2    25        55.40       77.56      22.16   41252.59 241252.59       0.00  Target Hit
      2 2024-01-19   0.509    6    25       124.50      105.83     -18.67  -35455.07 205797.52      14.70   Stop Loss
      3 2024-02-02   0.485    6    21       128.70      109.39     -19.31  -30809.17 174988.35      27.47   Stop Loss
      4 2024-02-07   0.486    1    25        67.45       94.43      26.98   50235.66 225224.01       6.64  Target Hit
      5 2024-04-04   0.509    0    10        43